# 04 — Ablation Study

Recreates the validation layer ablation (Table 5 / Table 8) and Figure 13 from the paper.

**Sections:**
1. Validation layer ablation — remove one layer at a time
2. Component ablation — RAG, SC, and both
3. SC rounds ablation (0, 1, 2, 3, 5 rounds)
4. RAG top-k ablation (k = 1, 3, 5, 10)
5. Embedding model ablation (MiniLM vs BGE-Large)
6. Prompt template ablation (base vs structured vs chain-of-thought)
7. Table 8 — Full ablation summary

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
FIGURES_DIR = Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

_results_path = Path('../results/experiment_results.json')
DATA = json.loads(_results_path.read_text()) if _results_path.exists() else {}
print('Results loaded.' if DATA else 'Using paper reference values.')

BASELINE_FUNC = 69.4  # DeepSeek, sc+rag+3r
BASELINE_SEC  = 59.3

## 1. Validation Layer Ablation

In [ ]:
layer_ablation = pd.DataFrame([
    {'Config':                'All layers (baseline)',   'func': 69.4, 'sec': 59.3, 'Δfunc': 0.0,  'Δsec': 0.0},
    {'Config':                '− L4 best practices',    'func': 69.8, 'sec': 58.1, 'Δfunc': +0.4, 'Δsec': -1.2},
    {'Config':                '− L3 security',          'func': 70.1, 'sec': 39.7, 'Δfunc': +0.7, 'Δsec':-19.6},
    {'Config':                '− L2.5 kubectl dry-run', 'func': 66.3, 'sec': 56.8, 'Δfunc': -3.1, 'Δsec': -2.5},
    {'Config':                '− L2 schema',            'func': 61.2, 'sec': 52.4, 'Δfunc': -8.2, 'Δsec': -6.9},
    {'Config':                '− L1 syntax',            'func': 54.8, 'sec': 47.1, 'Δfunc':-14.6, 'Δsec':-12.2},
    {'Config':                'No validation layers',   'func': 52.1, 'sec': 39.7, 'Δfunc':-17.3, 'Δsec':-19.6},
])

# Use actual data if available
ablation_data = DATA.get('ablation', {}).get('layers', {})
for i, row in layer_ablation.iterrows():
    key = row['Config'].lower().replace(' ', '_').replace('.', '').replace('−_', 'no_')
    if key in ablation_data:
        layer_ablation.at[i, 'func'] = ablation_data[key].get('functional_accuracy', row['func'])
        layer_ablation.at[i, 'sec']  = ablation_data[key].get('security_pass_rate',  row['sec'])

print('Layer Ablation Results:')
print(layer_ablation.to_string(index=False))

x = np.arange(len(layer_ablation))
w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, layer_ablation['func'], w, label='Functional', color='#4472C4', alpha=0.85)
ax.bar(x + w/2, layer_ablation['sec'],  w, label='Security',   color='#ED7D31', alpha=0.85)
ax.axhline(y=BASELINE_FUNC, color='#4472C4', linestyle=':', linewidth=1, alpha=0.6)
ax.axhline(y=BASELINE_SEC,  color='#ED7D31', linestyle=':', linewidth=1, alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(layer_ablation['Config'], fontsize=8, rotation=15, ha='right')
ax.set_ylabel('Pass Rate (%)')
ax.set_title('Validation Layer Ablation — DeepSeek Coder v2, sc+rag+3r')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_layers.pdf', bbox_inches='tight')
plt.show()

## 2. Component Ablation (RAG, SC)

In [ ]:
component_ablation = pd.DataFrame([
    {'Config': 'one-shot\n(no RAG, no SC)', 'func': 52.1, 'sec': 39.7},
    {'Config': '+RAG only',                 'func': 57.3, 'sec': 59.6},
    {'Config': '+SC only\n(3 rounds)',       'func': 63.8, 'sec': 58.1},
    {'Config': '+RAG+SC\n(sc+rag+3r)',       'func': 69.4, 'sec': 59.3},
    {'Config': '+RAG+SC\n(sc+rag+5r)',       'func': 71.2, 'sec': 60.8},
])

# Contributions
rag_func_gain = component_ablation.loc[1,'func'] - component_ablation.loc[0,'func']
rag_sec_gain  = component_ablation.loc[1,'sec']  - component_ablation.loc[0,'sec']
sc_func_gain  = component_ablation.loc[2,'func'] - component_ablation.loc[0,'func']
sc_sec_gain   = component_ablation.loc[2,'sec']  - component_ablation.loc[0,'sec']

x = np.arange(len(component_ablation))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - w/2, component_ablation['func'], w, label='Functional', color='#4472C4', alpha=0.85)
ax.bar(x + w/2, component_ablation['sec'],  w, label='Security',   color='#ED7D31', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(component_ablation['Config'], fontsize=9)
ax.set_ylabel('Pass Rate (%)')
ax.set_title('Component Ablation: RAG and SC Contributions')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_components.pdf', bbox_inches='tight')
plt.show()

print(f'RAG functional gain:  +{rag_func_gain:.1f}pp  |  RAG security gain:  +{rag_sec_gain:.1f}pp')
print(f'SC functional gain:   +{sc_func_gain:.1f}pp  |  SC security gain:   +{sc_sec_gain:.1f}pp')
print(f'Prevention-over-repair: RAG security gain ({rag_sec_gain:.1f}pp) > SC security gain ({sc_sec_gain:.1f}pp)')

## 3. SC Rounds Ablation (0 → 5)

In [ ]:
rounds_ablation = pd.DataFrame([
    {'rounds': 0, 'func': 57.3, 'sec': 59.6, 'total_time_min': 11.2},
    {'rounds': 1, 'func': 62.1, 'sec': 59.3, 'total_time_min': 18.4},
    {'rounds': 2, 'func': 67.4, 'sec': 59.4, 'total_time_min': 24.1},
    {'rounds': 3, 'func': 69.4, 'sec': 59.3, 'total_time_min': 28.6},
    {'rounds': 5, 'func': 71.2, 'sec': 60.8, 'total_time_min': 37.9},
])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.plot(rounds_ablation['rounds'], rounds_ablation['func'], marker='o',
         color='#4472C4', linewidth=2, label='Functional accuracy')
ax1.plot(rounds_ablation['rounds'], rounds_ablation['sec'],  marker='s',
         color='#ED7D31', linewidth=2, label='Security pass rate')
ax2.plot(rounds_ablation['rounds'], rounds_ablation['total_time_min'],
         marker='^', color='gray', linewidth=1.5, linestyle='--', label='Total time (min)')

ax1.set_xlabel('Max SC Rounds')
ax1.set_ylabel('Pass Rate (%)')
ax2.set_ylabel('Total Experiment Time (min)', color='gray')
ax1.set_title('SC Rounds Ablation: Accuracy vs Cost (DeepSeek, +RAG)')
ax1.set_xticks(rounds_ablation['rounds'])
ax1.set_ylim(50, 80)
ax1.grid(axis='y', linestyle='--', alpha=0.4)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_sc_rounds.pdf', bbox_inches='tight')
plt.show()

# Marginal gain analysis
for i in range(1, len(rounds_ablation)):
    cur  = rounds_ablation.iloc[i]
    prev = rounds_ablation.iloc[i-1]
    extra_t = cur['total_time_min'] - prev['total_time_min']
    gain    = cur['func'] - prev['func']
    print(f'Round {int(prev["rounds"])}→{int(cur["rounds"])}: +{gain:.1f}pp functional, +{extra_t:.1f} min')

## 4. RAG Top-k Ablation

In [ ]:
topk_ablation = pd.DataFrame([
    {'k': 1,  'p_at_k': 0.74, 'func': 54.8, 'sec': 56.1, 'ctx_tokens': 312},
    {'k': 3,  'p_at_k': 0.78, 'func': 58.9, 'sec': 58.3, 'ctx_tokens': 934},
    {'k': 5,  'p_at_k': 0.74, 'func': 69.4, 'sec': 59.3, 'ctx_tokens': 1556},
    {'k': 10, 'p_at_k': 0.69, 'func': 68.1, 'sec': 58.7, 'ctx_tokens': 3112},
])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(topk_ablation['k'], topk_ablation['func'], marker='o', linewidth=2,
        color='#4472C4', label='Functional')
ax.plot(topk_ablation['k'], topk_ablation['sec'],  marker='s', linewidth=2,
        color='#ED7D31', label='Security')
ax.set_xlabel('RAG top-k')
ax.set_ylabel('Pass Rate (%)')
ax.set_title('Accuracy vs RAG top-k')
ax.set_xticks(topk_ablation['k'])
ax.legend()
ax.grid(linestyle='--', alpha=0.4)

ax = axes[1]
ax.scatter(topk_ablation['ctx_tokens'], topk_ablation['func'],
           s=80, color='#4472C4', zorder=3)
for _, row in topk_ablation.iterrows():
    ax.annotate(f'k={int(row["k"])}', (row['ctx_tokens'], row['func']),
                textcoords='offset points', xytext=(5, 4), fontsize=9)
ax.set_xlabel('Context Tokens Added by RAG')
ax.set_ylabel('Functional Accuracy (%)')
ax.set_title('Context Size vs Accuracy')
ax.grid(linestyle='--', alpha=0.4)

plt.suptitle('RAG Top-k Ablation', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_topk.pdf', bbox_inches='tight')
plt.show()
print('Optimal k=5: best functional accuracy with reasonable context size')
print('k=10 shows slight degradation — context length dilution effect')

## 5. Embedding Model Ablation

In [ ]:
embed_ablation = pd.DataFrame([
    {'model': 'all-MiniLM-L6-v2\n(paper default)', 'dim': 384,  'params_m': 23,
     'p5_avg': 0.74, 'func': 69.4, 'sec': 59.3, 'embed_ms': 8},
    {'model': 'all-mpnet-base-v2',                 'dim': 768,  'params_m': 110,
     'p5_avg': 0.76, 'func': 70.1, 'sec': 60.1, 'embed_ms': 21},
    {'model': 'BAAI/bge-large-en-v1.5\n(+2.3pp)',  'dim': 1024, 'params_m': 335,
     'p5_avg': 0.77, 'func': 71.7, 'sec': 61.6, 'embed_ms': 47},
    {'model': 'text-embedding-3-small\n(OpenAI)',  'dim': 1536, 'params_m': None,
     'p5_avg': 0.78, 'func': 72.3, 'sec': 62.4, 'embed_ms': 84},
])

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics   = [('p5_avg', 'P@5 (retrieval quality)'),
             ('func',   'Functional Accuracy (%)'),
             ('sec',    'Security Pass Rate (%)')]

for ax, (col, title) in zip(axes, metrics):
    bars = ax.bar(range(len(embed_ablation)), embed_ablation[col],
                  color='#4472C4', alpha=0.85)
    ax.set_xticks(range(len(embed_ablation)))
    ax.set_xticklabels(embed_ablation['model'], fontsize=7.5, rotation=10)
    ax.set_title(title, fontsize=9)
    for bar, val in zip(bars, embed_ablation[col]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.003 if col == 'p5_avg' else val + 0.1,
                f'{val:.2f}' if col == 'p5_avg' else f'{val:.1f}',
                ha='center', fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Embedding Model Ablation (sc+rag+3r, DeepSeek)', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_embedding.pdf', bbox_inches='tight')
plt.show()
print('BGE-Large: +2.3pp security vs MiniLM, at 6× higher embedding latency')
print('MiniLM chosen for paper: best accuracy/latency trade-off for CPU-only inference')

## 6. Prompt Template Ablation

In [ ]:
prompt_ablation = pd.DataFrame([
    {'template': 'Base\n(task only)',             'func': 52.1, 'sec': 39.7},
    {'template': 'Base+RAG context',              'func': 57.3, 'sec': 59.6},
    {'template': 'Structured\n(task+RAG+format)', 'func': 69.4, 'sec': 59.3},
    {'template': 'CoT\n(chain-of-thought)',        'func': 67.8, 'sec': 57.4},
    {'template': 'Structured+CoT',                'func': 70.1, 'sec': 59.8},
])

x = np.arange(len(prompt_ablation))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - w/2, prompt_ablation['func'], w, label='Functional', color='#4472C4', alpha=0.85)
ax.bar(x + w/2, prompt_ablation['sec'],  w, label='Security',   color='#ED7D31', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(prompt_ablation['template'], fontsize=9)
ax.set_ylabel('Pass Rate (%)')
ax.set_title('Prompt Template Ablation')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_prompt.pdf', bbox_inches='tight')
plt.show()
print('Structured template (paper default) best balance: easy to parse, no extra latency')
print('CoT adds minimal gain (+0.7pp) at cost of longer outputs and higher API cost')

## 7. Table 8 — Full Ablation Summary

In [ ]:
table8 = pd.DataFrame([
    # Component ablation
    {'Category': 'Component',  'Variant': 'one-shot (baseline)',        'func': 52.1, 'sec': 39.7, 'Δfunc': 0.0,  'Δsec': 0.0},
    {'Category': 'Component',  'Variant': '+RAG only',                  'func': 57.3, 'sec': 59.6, 'Δfunc': +5.2, 'Δsec':+19.9},
    {'Category': 'Component',  'Variant': '+SC (3r) only',              'func': 63.8, 'sec': 58.1, 'Δfunc':+11.7, 'Δsec':+18.4},
    {'Category': 'Component',  'Variant': '+RAG+SC (sc+rag+3r)',        'func': 69.4, 'sec': 59.3, 'Δfunc':+17.3, 'Δsec':+19.6},
    # Layer ablation (relative to sc+rag+3r)
    {'Category': 'Layer',      'Variant': '− L3 security layer',        'func': 70.1, 'sec': 39.7, 'Δfunc': +0.7, 'Δsec':-19.6},
    {'Category': 'Layer',      'Variant': '− L2.5 kubectl dry-run',     'func': 66.3, 'sec': 56.8, 'Δfunc': -3.1, 'Δsec': -2.5},
    {'Category': 'Layer',      'Variant': '− L2 schema',                'func': 61.2, 'sec': 52.4, 'Δfunc': -8.2, 'Δsec': -6.9},
    # SC rounds
    {'Category': 'SC rounds',  'Variant': '0 rounds (+RAG)',            'func': 57.3, 'sec': 59.6, 'Δfunc':-12.1, 'Δsec': +0.3},
    {'Category': 'SC rounds',  'Variant': '1 round  (+RAG)',            'func': 62.1, 'sec': 59.3, 'Δfunc': -7.3, 'Δsec':  0.0},
    {'Category': 'SC rounds',  'Variant': '3 rounds (+RAG) [default]', 'func': 69.4, 'sec': 59.3, 'Δfunc':  0.0, 'Δsec':  0.0},
    {'Category': 'SC rounds',  'Variant': '5 rounds (+RAG)',            'func': 71.2, 'sec': 60.8, 'Δfunc': +1.8, 'Δsec': +1.5},
    # RAG k
    {'Category': 'RAG top-k',  'Variant': 'k=1',                       'func': 54.8, 'sec': 56.1, 'Δfunc':-14.6, 'Δsec': -3.2},
    {'Category': 'RAG top-k',  'Variant': 'k=3',                       'func': 58.9, 'sec': 58.3, 'Δfunc':-10.5, 'Δsec': -1.0},
    {'Category': 'RAG top-k',  'Variant': 'k=5 [default]',             'func': 69.4, 'sec': 59.3, 'Δfunc':  0.0, 'Δsec':  0.0},
    {'Category': 'RAG top-k',  'Variant': 'k=10',                      'func': 68.1, 'sec': 58.7, 'Δfunc': -1.3, 'Δsec': -0.6},
    # Embedding model
    {'Category': 'Embedding',  'Variant': 'all-MiniLM-L6-v2 [default]','func': 69.4, 'sec': 59.3, 'Δfunc':  0.0, 'Δsec':  0.0},
    {'Category': 'Embedding',  'Variant': 'bge-large-en-v1.5',         'func': 71.7, 'sec': 61.6, 'Δfunc': +2.3, 'Δsec': +2.3},
])

print('Table 8 — Full Ablation Study Summary')
print('=' * 80)
for cat, grp in table8.groupby('Category', sort=False):
    print(f'\n{cat}:')
    print(grp[['Variant', 'func', 'sec', 'Δfunc', 'Δsec']].to_string(index=False))

# Save CSV
csv_path = Path('../results/table8_ablation.csv')
table8.to_csv(csv_path, index=False)
print(f'\nSaved to {csv_path}')